# Notebook 10: Observability & LangGraph Platform

## Shipping LangGraph to Production

This is the final notebook in the series. We cover the production lifecycle from observability to deployment:

1. **LangSmith** — Tracing, evaluation, and monitoring
2. **LangGraph Platform** — Deployment, REST APIs, remote graphs
3. **LangGraph Studio v2** — Local debugging and time-travel

**Prerequisites**: Notebooks 01-09  
**Time**: 30-40 minutes  
**LangGraph Version**: >=1.1.9

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

import langgraph
#print(f"LangGraph: {langgraph.__version__}")

# Check optional dependencies
try:
    import langgraph_sdk
    print(f"langgraph-sdk: installed")
except ImportError:
    print("Note: langgraph-sdk not installed. Install with: uv pip install langgraph-sdk>=0.3.13")

print(f"\nLangSmith configured: {bool(os.getenv('LANGSMITH_API_KEY'))}")
print(f"Tracing enabled: {os.getenv('LANGCHAIN_TRACING_V2', 'false').lower() == 'true'}")

langgraph-sdk: installed

LangSmith configured: True
Tracing enabled: False


---
## Setup: Demo Graph

We'll use a simple support ticket classifier throughout this notebook to demonstrate observability and evaluation. This keyword-based router requires no API keys and produces predictable, deterministic outputs — ideal for LangSmith tracing demos and building evaluation datasets.

In [2]:
import sqlite3
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# ── State schema ─────────────────────────────────────────────────────────────
class TicketState(TypedDict):
    message: str
    category: str
    priority: str
    response: str

# ── Node: keyword-based classifier (no LLM needed) ───────────────────────────
def classify_ticket(state: TicketState) -> dict:
    """Route support ticket by keyword matching."""
    msg = state["message"].lower()
    if any(w in msg for w in ["billing", "charge", "refund", "payment", "invoice"]):
        return {"category": "billing", "priority": "high"}
    elif any(w in msg for w in ["crash", "error", "bug", "broken", "not working"]):
        return {"category": "technical", "priority": "high"}
    elif any(w in msg for w in ["feature", "request", "suggest", "would be nice", "add"]):
        return {"category": "feature", "priority": "low"}
    else:
        return {"category": "general", "priority": "medium"}

# ── Node: template-based response generator ───────────────────────────────────
def format_response(state: TicketState) -> dict:
    """Return a canned response based on ticket category."""
    templates = {
        "billing":   "Our billing team will review your issue within 24 hours. Please have your account ID ready.",
        "technical": "Our engineering team has been notified. Expect an update within 4 hours.",
        "feature":   "Thank you for the suggestion! We have added it to our product roadmap.",
        "general":   "Our support team will respond within 2 business days.",
    }
    return {"response": templates.get(state["category"], "Thank you for contacting us.")}

# ── Build and compile ─────────────────────────────────────────────────────────
builder = StateGraph(TicketState)
builder.add_node("classify", classify_ticket)
builder.add_node("respond", format_response)
builder.add_edge(START, "classify")
builder.add_edge("classify", "respond")
builder.add_edge("respond", END)

app = builder.compile(checkpointer=MemorySaver())

# ── Quick smoke test ──────────────────────────────────────────────────────────
_result = app.invoke(
    {"message": "I was charged twice this month", "category": "", "priority": "", "response": ""},
    {"configurable": {"thread_id": "setup-verify-001"}}
)
print("\u2705 Demo graph compiled and verified")
print(f"   Input:    'I was charged twice this month'")
print(f"   Category: {_result['category']} | Priority: {_result['priority']}")
print(f"   Response: {_result['response']}")

✅ Demo graph compiled and verified
   Input:    'I was charged twice this month'
   Category: billing | Priority: high
   Response: Our billing team will review your issue within 24 hours. Please have your account ID ready.


---
# Part 1: LangSmith — Observability & Evaluation

## Setup

LangSmith traces ALL LangGraph runs automatically with zero code changes:

```bash
# Add to .env:
LANGSMITH_API_KEY=ls-your-key-here
LANGCHAIN_TRACING_V2=true
LANGCHAIN_PROJECT=my-project-name
```

Every `.invoke()`, `.stream()`, and `.ainvoke()` call is automatically captured.

In [3]:
import os

# Check LangSmith configuration
langsmith_api_key = os.getenv("LANGSMITH_API_KEY")
tracing_enabled = os.getenv("LANGCHAIN_TRACING_V2", "false").lower() == "true"
project = os.getenv("LANGCHAIN_PROJECT", "default")

print("=== LangSmith Configuration ===")
print(f"API Key: {'\u2705 Configured' if langsmith_api_key else '\u274c Not set'}")
print(f"Tracing: {'\u2705 Enabled' if tracing_enabled else '\u274c Disabled'}")
print(f"Project: {project}")

if not langsmith_api_key:
    print("\nTo enable LangSmith tracing:")
    print("1. Create account at https://smith.langchain.com")
    print("2. Get your API key from Settings > API Keys")
    print("3. Add to .env:")
    print("   LANGSMITH_API_KEY=ls-your-key-here")
    print("   LANGCHAIN_TRACING_V2=true")
    print("   LANGCHAIN_PROJECT=langgraph-tutorials")

# Adding metadata to traced runs
print("\n=== Adding Custom Metadata ===")
config_with_metadata = {
    "configurable": {"thread_id": "demo-001"},
    "tags": ["production", "v2.0", "customer-support"],
    "metadata": {
        "user_id": "user-42",
        "session_type": "support",
        "environment": "production",
        "feature_flags": ["new_routing_v2"],
    },
    "run_name": "Customer Support - Ticket Classification",
}
print("Run config with full LangSmith metadata:")
for k, v in config_with_metadata.items():
    print(f"  {k}: {v}")

=== LangSmith Configuration ===
API Key: ✅ Configured
Tracing: ❌ Disabled
Project: langgraph-advanced

=== Adding Custom Metadata ===
Run config with full LangSmith metadata:
  configurable: {'thread_id': 'demo-001'}
  tags: ['production', 'v2.0', 'customer-support']
  metadata: {'user_id': 'user-42', 'session_type': 'support', 'environment': 'production', 'feature_flags': ['new_routing_v2']}
  run_name: Customer Support - Ticket Classification


In [4]:
# ============================================================
# LangSmith Evaluation — 4 Concrete Examples
# ============================================================
# All examples run locally without a LangSmith account.
# When LANGSMITH_API_KEY is set, replace the local loops with
# langsmith.evaluation.evaluate() for scale and persistence.

import os

# ── Shared test dataset ───────────────────────────────────────────────────────
TEST_CASES = [
    {"message": "I need a refund for my subscription",  "expected_category": "billing",   "expected_priority": "high"},
    {"message": "The app keeps crashing on iOS 17",     "expected_category": "technical",  "expected_priority": "high"},
    {"message": "Would love to see dark mode added",    "expected_category": "feature",    "expected_priority": "low"},
    {"message": "How do I reset my password?",          "expected_category": "general",    "expected_priority": "medium"},
    {"message": "I was charged twice this month",       "expected_category": "billing",    "expected_priority": "high"},
    {"message": "Login button is broken on mobile",     "expected_category": "technical",  "expected_priority": "high"},
]

def _run(message: str, thread_id: str) -> dict:
    return app.invoke(
        {"message": message, "category": "", "priority": "", "response": ""},
        {"configurable": {"thread_id": thread_id}}
    )

# ─────────────────────────────────────────────────────────────────────────────
# EXAMPLE 1: Classification Accuracy
# → Did the classifier assign the correct category?
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 62)
print("EXAMPLE 1: Classification Accuracy")
print("=" * 62)

def eval_classification(result: dict, expected_category: str) -> dict:
    predicted = result["category"]
    correct = predicted == expected_category
    return {"score": 1.0 if correct else 0.0,
            "label": "correct" if correct else "wrong",
            "detail": f"predicted={predicted!r}  expected={expected_category!r}"}

hits = 0
for i, tc in enumerate(TEST_CASES):
    r = _run(tc["message"], f"ex1-{i}")
    ev = eval_classification(r, tc["expected_category"])
    hits += ev["score"]
    icon = "\u2705" if ev["label"] == "correct" else "\u274c"
    print(f"  {icon} {tc['message'][:45]:<45} {ev['detail']}")

print(f"\nAccuracy: {hits/len(TEST_CASES):.1%}  ({int(hits)}/{len(TEST_CASES)})")

# ─────────────────────────────────────────────────────────────────────────────
# EXAMPLE 2: Response Completeness
# → Does the response contain required keywords for its category?
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 62)
print("EXAMPLE 2: Response Completeness")
print("=" * 62)

REQUIRED_KEYWORDS = {
    "billing":   ["24 hours", "billing"],
    "technical": ["engineering", "hours"],
    "feature":   ["thank", "roadmap"],
    "general":   ["support", "days"],
}

def eval_completeness(result: dict) -> dict:
    required = REQUIRED_KEYWORDS.get(result["category"], [])
    response = result["response"].lower()
    hits = sum(1 for kw in required if kw in response)
    score = hits / len(required) if required else 1.0
    return {"score": score, "hits": hits, "total": len(required)}

total = 0.0
for i, tc in enumerate(TEST_CASES):
    r = _run(tc["message"], f"ex2-{i}")
    ev = eval_completeness(r)
    total += ev["score"]
    icon = "\u2705" if ev["score"] == 1.0 else "\u26a0\ufe0f"
    print(f"  {icon} [{r['category']:<10}] keywords={ev['hits']}/{ev['total']}  {r['response'][:55]}")

print(f"\nAvg Completeness: {total/len(TEST_CASES):.1%}")

# ─────────────────────────────────────────────────────────────────────────────
# EXAMPLE 3: Tone Check (Custom Metric)
# → Is the response polite and professional?
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 62)
print("EXAMPLE 3: Tone Check  (Custom Heuristic Metric)")
print("=" * 62)

POLITE = ["thank", "please", "appreciate", "sorry", "will", "review", "notified", "team"]
COLD   = ["cannot", "impossible", "never", "refuse"]

def eval_tone(result: dict) -> dict:
    resp = result["response"].lower()
    polite_hits = sum(1 for p in POLITE if p in resp)
    cold_hits   = sum(1 for p in COLD   if p in resp)
    raw = min(1.0, polite_hits / 2) * (0.5 if cold_hits > 0 else 1.0)
    label = "professional" if raw >= 0.7 else "neutral" if raw >= 0.4 else "cold"
    return {"score": round(raw, 2), "tone": label}

for i, tc in enumerate(TEST_CASES[:4]):
    r = _run(tc["message"], f"ex3-{i}")
    ev = eval_tone(r)
    icon = "\u2705" if ev["tone"] == "professional" else "\u26a0\ufe0f"
    print(f"  {icon} [{r['category']:<10}] tone={ev['tone']:<14} score={ev['score']:.2f}")

print("\nCustom metrics let you encode domain-specific quality signals beyond correctness.")

# ─────────────────────────────────────────────────────────────────────────────
# EXAMPLE 4: A/B Experiment — Compare Two Routing Strategies
# → Measure accuracy of current classifier vs an improved v2
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 62)
print("EXAMPLE 4: A/B Experiment  (Strategy v1 vs v2)")
print("=" * 62)

def classify_ticket_v2(state: TicketState) -> dict:
    """Strategy v2 — broader keyword coverage and edge-case handling."""
    msg = state["message"].lower()
    if any(w in msg for w in ["billing", "charge", "refund", "payment", "invoice",
                               "overcharge", "money back", "cancel"]):
        return {"category": "billing", "priority": "high"}
    elif any(w in msg for w in ["crash", "error", "bug", "broken", "not working",
                                 "slow", "down", "outage", "not loading", "fail"]):
        return {"category": "technical", "priority": "high"}
    elif any(w in msg for w in ["feature", "request", "suggest", "would be nice", "add",
                                  "wish", "improve", "better if", "option to"]):
        return {"category": "feature", "priority": "low"}
    else:
        return {"category": "general", "priority": "medium"}

builder_b = StateGraph(TicketState)
builder_b.add_node("classify", classify_ticket_v2)
builder_b.add_node("respond", format_response)
builder_b.add_edge(START, "classify")
builder_b.add_edge("classify", "respond")
builder_b.add_edge("respond", END)
app_v2 = builder_b.compile(checkpointer=MemorySaver())

# Include edge cases that v1 misses
AB_CASES = [
    {"message": "I want my money back",          "expected_category": "billing"},
    {"message": "App is extremely slow today",   "expected_category": "technical"},
    {"message": "I wish there was a dark mode",  "expected_category": "feature"},
    {"message": "Site is completely down!",      "expected_category": "technical"},
    *[{"message": tc["message"], "expected_category": tc["expected_category"]} for tc in TEST_CASES[:3]],
]

ok_a, ok_b = [], []
for i, tc in enumerate(AB_CASES):
    ra = _run(tc["message"], f"ex4a-{i}")
    rb = app_v2.invoke(
        {"message": tc["message"], "category": "", "priority": "", "response": ""},
        {"configurable": {"thread_id": f"ex4b-{i}"}}
    )
    ca, cb = ra["category"] == tc["expected_category"], rb["category"] == tc["expected_category"]
    ok_a.append(ca); ok_b.append(cb)
    change = "  same" if ca == cb else ("  \u2191 improved" if cb else "  \u2193 regressed")
    print(f"  {tc['message'][:45]:<45} v1={'\u2705' if ca else '\u274c'} v2={'\u2705' if cb else '\u274c'}{change}")

acc_a, acc_b = sum(ok_a)/len(ok_a), sum(ok_b)/len(ok_b)
delta = acc_b - acc_a
print(f"\nv1 accuracy: {acc_a:.1%}  |  v2 accuracy: {acc_b:.1%}  |  delta: {'+' if delta >= 0 else ''}{delta:.1%}")
print("\n\U0001f4a1 LangSmith SDK pattern (requires account):")
print("""
    from langsmith.evaluation import evaluate
    results = evaluate(
        lambda inputs: app_v2.invoke({**inputs, 'category': '', 'priority': '', 'response': ''}),
        data='ticket-classification-eval-dataset',
        evaluators=[eval_classification, eval_completeness, eval_tone],
        experiment_prefix='classifier-v2',
    )
""")

EXAMPLE 1: Classification Accuracy
  ✅ I need a refund for my subscription           predicted='billing'  expected='billing'
  ✅ The app keeps crashing on iOS 17              predicted='technical'  expected='technical'
  ✅ Would love to see dark mode added             predicted='feature'  expected='feature'
  ✅ How do I reset my password?                   predicted='general'  expected='general'
  ✅ I was charged twice this month                predicted='billing'  expected='billing'
  ✅ Login button is broken on mobile              predicted='technical'  expected='technical'

Accuracy: 100.0%  (6/6)

EXAMPLE 2: Response Completeness
  ✅ [billing   ] keywords=2/2  Our billing team will review your issue within 24 hours
  ✅ [technical ] keywords=2/2  Our engineering team has been notified. Expect an updat
  ✅ [feature   ] keywords=2/2  Thank you for the suggestion! We have added it to our p
  ✅ [general   ] keywords=2/2  Our support team will respond within 2 business days.
  ✅ [billing

---
# Part 2: LangGraph Platform

## Deployment Options

| Tier | Description | Best for |
|---|---|---|
| **Developer** | Free, 100K node executions/month | Learning, prototypes |
| **Cloud SaaS** | Fully managed, horizontal scaling | Most production apps |
| **Hybrid** | SaaS control plane + your data plane | Compliance requirements |
| **Self-Hosted** | Full control, your infrastructure | Maximum control |

## Deployment Steps (Cloud SaaS)

1. Create `langgraph.json` in your project root
2. Connect your GitHub repository in LangSmith
3. Deploy with 1 click
4. Access via REST API or Remote Graph SDK

In [5]:
# langgraph.json — deployment configuration
langgraph_config = {
    "dependencies": ["."],
    "graphs": {
        "support_agent": "./src/agent.py:app",     # module_path:variable_name
        "research_agent": "./src/research.py:app",
    },
    "env": ".env",
    "python_version": "3.12",
}

import json
print("=== langgraph.json (deployment config) ===")
print(json.dumps(langgraph_config, indent=2))

print("\nDeploy commands:")
print("  langgraph build          # Build Docker image")
print("  langgraph up             # Run locally (for testing)")
print("  langgraph deploy         # Deploy to LangGraph Platform")

=== langgraph.json (deployment config) ===
{
  "dependencies": [
    "."
  ],
  "graphs": {
    "support_agent": "./src/agent.py:app",
    "research_agent": "./src/research.py:app"
  },
  "env": ".env",
  "python_version": "3.12"
}

Deploy commands:
  langgraph build          # Build Docker image
  langgraph up             # Run locally (for testing)
  langgraph deploy         # Deploy to LangGraph Platform


In [6]:
# Remote Graph — use deployed graph as local object
print("=== Remote Graph Pattern ===\n")

remote_graph_example = '''
from langgraph_sdk import RemoteGraph
from langchain_core.messages import HumanMessage

# Connect to deployed graph — same API as local graph!
remote_app = RemoteGraph(
    "support_agent",                                    # graph name from langgraph.json
    url="https://your-deployment.langchain.com",       # deployment URL
    api_key=os.getenv("LANGGRAPH_API_KEY"),
)

# Use exactly like a local graph
result = await remote_app.ainvoke(
    {"messages": [HumanMessage("I have a billing issue")]},
    config={"configurable": {"thread_id": "user-001"}}
)

# Stream from remote graph
async for chunk in remote_app.astream(
    {"messages": [HumanMessage("Help me with my account")]},
    stream_mode="messages"
):
    print(chunk)
'''

print("Remote Graph usage (requires LangGraph Platform deployment):")
print(remote_graph_example)

=== Remote Graph Pattern ===

Remote Graph usage (requires LangGraph Platform deployment):

from langgraph_sdk import RemoteGraph
from langchain_core.messages import HumanMessage

# Connect to deployed graph — same API as local graph!
remote_app = RemoteGraph(
    "support_agent",                                    # graph name from langgraph.json
    url="https://your-deployment.langchain.com",       # deployment URL
    api_key=os.getenv("LANGGRAPH_API_KEY"),
)

# Use exactly like a local graph
result = await remote_app.ainvoke(
    {"messages": [HumanMessage("I have a billing issue")]},
    config={"configurable": {"thread_id": "user-001"}}
)

# Stream from remote graph
async for chunk in remote_app.astream(
    {"messages": [HumanMessage("Help me with my account")]},
    stream_mode="messages"
):
    print(chunk)



In [7]:
# REST API endpoints available on LangGraph Platform
print("=== LangGraph Platform REST API ===\n")

api_endpoints = {
    "POST /runs": "Start a synchronous run",
    "POST /runs/stream": "Start a streaming run",
    "POST /runs/background": "Fire-and-forget async run",
    "POST /runs/wait": "Start run and wait for completion",
    "POST /threads": "Create a new conversation thread",
    "GET /threads/{id}/state": "Get current thread state",
    "POST /threads/{id}/state": "Update thread state",
    "GET /threads/{id}/history": "Get full state history",
    "POST /runs/crons": "Schedule recurring runs",
    "POST /webhooks": "Register webhook for run completion",
    "GET /assistants": "List available graph assistants",
    "POST /assistants/{id}/runs": "Run a specific assistant version",
}

print("Available REST API endpoints:")
for endpoint, description in api_endpoints.items():
    print(f"  {endpoint:<35} \u2192 {description}")

print("\n\U0001f4a1 Tip: Use Remote Graph SDK instead of raw REST API when possible — ")
print("    it handles authentication, serialization, and streaming automatically.")

=== LangGraph Platform REST API ===

Available REST API endpoints:
  POST /runs                          → Start a synchronous run
  POST /runs/stream                   → Start a streaming run
  POST /runs/background               → Fire-and-forget async run
  POST /runs/wait                     → Start run and wait for completion
  POST /threads                       → Create a new conversation thread
  GET /threads/{id}/state             → Get current thread state
  POST /threads/{id}/state            → Update thread state
  GET /threads/{id}/history           → Get full state history
  POST /runs/crons                    → Schedule recurring runs
  POST /webhooks                      → Register webhook for run completion
  GET /assistants                     → List available graph assistants
  POST /assistants/{id}/runs          → Run a specific assistant version

💡 Tip: Use Remote Graph SDK instead of raw REST API when possible — 
    it handles authentication, serialization, and s

---
# Part 3: LangGraph Studio v2 — Local Debugging

## What's New in Studio v2

LangGraph Studio v2 is fully web-based — no desktop app required:

- **Run locally**: `langgraph serve --reload` launches a local server
- **Browser UI**: Open the Studio in any browser
- **Time-travel debugging**: Rewind to any checkpoint, edit state, fork execution
- **Production trace replay**: Pull real production traces and replay locally
- **Hot reloading**: Code changes reflect immediately
- **Subgraph inspection**: See inside nested subgraphs with `subgraphs=True`

In [8]:
print("=== LangGraph Studio v2 — Local Development ===\n")

print("1. Start local Studio:")
print("   langgraph serve --reload")
print("   # Opens at http://localhost:8123")
print()
print("2. Time-travel debugging:")
print("   - Run your graph")
print("   - Click any checkpoint in the history panel")
print("   - Edit state values directly in the UI")
print("   - Fork: create a new thread from that checkpoint")
print("   - Useful for: debugging, testing 'what if' scenarios")
print()
print("3. Replay production traces:")
print("   - Pull a failing trace from LangSmith")
print("   - Load it in Studio")
print("   - Step through node by node")
print("   - See exact state at each checkpoint")
print("   - Fix the issue locally, re-run to verify")
print()
print("4. Inspect subgraph execution:")
print("   # When streaming, add subgraphs=True:")
print('   for chunk in app.stream(input, config, stream_mode="updates", subgraphs=True):')
print("       # chunk['ns'] identifies which subgraph emitted this update")
print("       print(chunk['ns'], chunk['data'])")

# Demonstrate subgraph-aware streaming
from langgraph.graph import StateGraph, MessagesState, START, END
from typing_extensions import TypedDict

class ParentState(TypedDict):
    data: str
    processed: str

class ChildState(TypedDict):
    data: str
    result: str

# Build a subgraph
child_builder = StateGraph(ChildState)
child_builder.add_node("child_process", lambda s: {"result": f"[child] {s['data']}"})
child_builder.add_edge(START, "child_process")
child_builder.add_edge("child_process", END)
child_graph = child_builder.compile()

# Build parent graph using subgraph
parent_builder = StateGraph(ParentState)
parent_builder.add_node("preprocess", lambda s: {"data": s["data"].upper()})
parent_builder.add_node("child", child_graph)
parent_builder.add_node("postprocess", lambda s: {"processed": s.get("result", s["data"])})
parent_builder.add_edge(START, "preprocess")
parent_builder.add_edge("preprocess", "child")
parent_builder.add_edge("child", "postprocess")
parent_builder.add_edge("postprocess", END)

parent_app = parent_builder.compile()

print("\n=== Subgraph streaming with namespace info ===")
for chunk in parent_app.stream(
    {"data": "hello world", "processed": ""},
    stream_mode="updates",
    subgraphs=True  # \u2190 see inside subgraphs
):
    namespace, update = chunk  # (namespace_tuple, update_dict) when subgraphs=True
    source = "parent" if not namespace else f"subgraph({'.'.join(namespace)})"
    print(f"  [{source}] {update}")

=== LangGraph Studio v2 — Local Development ===

1. Start local Studio:
   langgraph serve --reload
   # Opens at http://localhost:8123

2. Time-travel debugging:
   - Run your graph
   - Click any checkpoint in the history panel
   - Edit state values directly in the UI
   - Fork: create a new thread from that checkpoint
   - Useful for: debugging, testing 'what if' scenarios

3. Replay production traces:
   - Pull a failing trace from LangSmith
   - Load it in Studio
   - Step through node by node
   - See exact state at each checkpoint
   - Fix the issue locally, re-run to verify

4. Inspect subgraph execution:
   # When streaming, add subgraphs=True:
   for chunk in app.stream(input, config, stream_mode="updates", subgraphs=True):
       # chunk['ns'] identifies which subgraph emitted this update
       print(chunk['ns'], chunk['data'])

=== Subgraph streaming with namespace info ===
  [parent] {'preprocess': {'data': 'HELLO WORLD'}}
  [subgraph(child:815ba8ec-39c1-82c6-a659-d55573

---
## Summary

### Part 1 — LangSmith: Observability & Evaluation
- Zero-code setup: set `LANGSMITH_API_KEY`, `LANGCHAIN_TRACING_V2=true`, `LANGCHAIN_PROJECT`
- Use `tags`, `metadata`, `run_name` in config for organized, searchable traces
- 4 evaluation patterns: **accuracy**, **completeness**, **custom metrics**, **A/B experiments**
- Build datasets once, run as regression tests on every release

### Part 2 — LangGraph Platform
- Start with the free Developer tier (100K node executions/month)
- Deploy via GitHub integration — 1 click to Cloud SaaS
- `RemoteGraph` SDK: identical API to local graphs, handles auth + serialization
- Key endpoints: `/runs`, `/runs/stream`, `/threads`, `/runs/crons`

### Part 3 — LangGraph Studio v2
- `langgraph serve --reload` for local dev with hot reloading
- Time-travel: rewind to any checkpoint, edit state, fork execution branches
- Replay production failures locally for step-by-step debugging
- `subgraphs=True` for full visibility into nested graph execution

---

## Congratulations — You've Completed the LangGraph Series! \U0001f389

1. \u2705 StateGraph fundamentals — nodes, edges, state schemas
2. \u2705 Conditional routing — decision-making graphs
3. \u2705 Tool-using agents — LLM + tool loop
4. \u2705 Persistence — MemorySaver, SqliteSaver, PostgresSaver
5. \u2705 Human-in-the-loop — interrupt() + Command(resume=)
6. \u2705 Parallel execution — Send API, subgraphs, deferred nodes
7. \u2705 Production patterns — caching, middleware, callbacks
8. \u2705 Command + Functional API + Long-term Memory
9. \u2705 Multi-Agent Systems — Supervisor, Swarm, custom architectures
10. \u2705 Observability & LangGraph Platform

**What's next:**
- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)
- [LangGraph GitHub](https://github.com/langchain-ai/langgraph)
- [LangSmith](https://smith.langchain.com)
- [LangGraph Platform](https://www.langchain.com/langgraph-platform)
- [LangGraph Discord](https://discord.gg/langchain)